# Generate SAP Mock Data
This notebook calls the packaged pandas generator and writes Delta tables.

In [ ]:
import os
from pathlib import Path

from sap_mock_data import GenerationConfig, Timeframe, generate_dataset
from sap_mock_data.storage import DeltaTableStore

warehouse = Path(os.environ.get('SAP_MOCK_WAREHOUSE', '.mock-warehouse')).resolve()
start_date = os.environ.get('SAP_MOCK_START_DATE', '')
end_date = os.environ.get('SAP_MOCK_END_DATE', '')
duration_days = os.environ.get('SAP_MOCK_DURATION_DAYS', '')
timeframe = Timeframe(start_date, end=end_date or None, duration_days=int(duration_days) if duration_days else None) if any((start_date, end_date, duration_days)) else None
scale = os.environ.get('SAP_MOCK_SCALE_FACTOR', '0.5')
scale = scale.upper() if scale.upper() in ('S', 'M', 'L', 'XL') else float(scale)
result = generate_dataset(
    GenerationConfig(
        timeframe=timeframe,
        random_seed=int(os.environ.get('SAP_MOCK_RANDOM_SEED', '42')),
        scale_factor=scale,
        scenarios=os.environ.get('SAP_MOCK_SCENARIOS', 'demo'),
    ),
    DeltaTableStore(warehouse),
)
result.table_count, result.row_counts